# 0. NOTEBOOK HEADER & METADATA

## Notebook Metadata

| Field | Value |
|---|---|
| **Target Table** | `nue.fact_inventorysnapshot` |
| **Product Group and Division** | To Be Updated By DJJ team |
| **Author** | Mohit Singhal |
| **Created** | 2026-05-13 |
| **DevOps ID and Link** | To Be Updated By DJJ team |
| **Load Type** | `` |
| **Grain** | To Be Updated By DJJ team |

---

## Description

To Be Updated By DJJ team

## Change Log

| Date | Author | DevOps ID and Link | Change Description |
|---|---|---|---|

# 1. CONFIGURATION, IMPORTS, & PARAMETERS

In [0]:
# ========================================
# LIBRARY IMPORTS
# ========================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
from delta.tables import DeltaTable
import uuid

In [0]:
# ==============================================
# CONFIGURATION
# ==============================================

federated_ods_catalog="djj_ods"                                    #Catalog name for source table
SourceSchemaName="dbo"                                          #Schema name for source table
SourceTableName="ODS_CORE_User_Demographic"                               #Table name for source table
enriched_catalog="djj_enriched_non_prod"                 #Catalog name for target table
DestinationSchemaName="nue"                                     #Schema name for target table
DestinationTableName="fact_InventorySnapshot_temp"                                  #Table name for target table
federated_starschema_catalog  = "djj_starschema"
etl_runtime = datetime.now()                                    #Current Runtime for updating LastUpdateTime

layer="enriched"                                                                            #Target layer
source_table=f"{federated_ods_catalog}.{SourceSchemaName}.{SourceTableName}"
target_table=f"{enriched_catalog}.{DestinationSchemaName}.{DestinationTableName}"
proc="incremental"                                                                    #Load type
platform="mssql"                                                                            #Source platform

# 2. START METADATA LOGGING

In [0]:
%skip
%run ../Metadata_Logger/execution_logger

In [0]:
%run "Workspace/Shared/DJJ Migration/Metadata_Logger/execution_logger"

In [0]:
# ===================================
# START METADATA LOGGING
# ===================================

run_ctx = start_execution_run(
    layer=layer,
    source_table=source_table,
    target_table=target_table,
    proc=proc,
    platform=platform
)

In [0]:
%sql
CREATE OR REPLACE TABLE djj_enriched_non_prod.NUE.fact_InventorySnapshot_temp (
    FiscalWeekKey                              INT NOT NULL,
    OrgID                                      INT NOT NULL,
    EnterpriseGradeGroup                       STRING COLLATE UTF8_LCASE NOT NULL,
    EnterpriseGradeGroupKey                    INT NOT NULL,
    ConsumerKey                                INT NOT NULL,
    ReceivedWgt                                BIGINT,
    ReceivedAmount                             DECIMAL(18, 2),
    BeginningWgt                               BIGINT,
    BeginningAmount                            DECIMAL(18, 4),
    ConsumedWgt                                BIGINT,
    ConsumedAmount                             DECIMAL(18, 4),
    EndingWgt                                  BIGINT,
    EndingAmount                               DECIMAL(18, 4),
    BeginningInTransitWgt                      INT,
    BeginningInTransitWgt_BeforeAdjustment     INT,
    BeginningInTransitAmount                   DECIMAL(18, 4),
    BeginningInTransitAmount_BeforeAdjustment  DECIMAL(18, 4),
    EndingInTransitWgt                         INT,
    EndingInTransitWgt_BeforeAdjustment        INT,
    EndingInTransitAmount                      DECIMAL(18, 4),
    EndingInTransitAmount_BeforeAdjustment     DECIMAL(18, 4),
    DJJGeneratedFlag                           INT NOT NULL,
    ETLSSExecutionID                           BIGINT,
    EnrichedTimestampUTC                          TIMESTAMP NOT NULL,
    CONSTRAINT PK_NUE_factInventorySnapshot PRIMARY KEY (FiscalWeekKey, OrgID, EnterpriseGradeGroup)
) USING DELTA;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7577138953583057>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REPLACE TABLE djj_enriched_non_prod.NUE.fact_InventorySnapshot_temp (\n    FiscalWeekKey                              INT NOT NULL,\n    OrgID                                      INT NOT NULL,\n    EnterpriseGradeGroup                       STRING COLLATE UTF8_LCASE NOT NULL,\n    EnterpriseGradeGroupKey                    INT NOT NULL,\n    ConsumerKey                                INT NOT NULL,\n    ReceivedWgt                                BIGINT,\n    ReceivedAmount                             DECIMAL(18, 2),\n    BeginningWgt                               BIGINT,\n    BeginningAmount                            DECIMAL(18, 4),\n    ConsumedWgt                                BIGINT,\n    ConsumedAmount                             DEC

In [0]:
%sql
insert into djj_enriched_non_prod.NUE.fact_InventorySnapshot_temp
select * from djj_starschema.NUE.factInventorySnapshot
where DJJLastUpdateTime <= '2026-05-10 22:53:22.983'

# 3. ETL Logic

In [0]:
# ===================================================
# OLE_SRC_rawMaterialReceipt - OLE DB Source
# ===================================================

ole_src_rawMaterialReceipt_output_df = spark.sql(f"""
    SELECT (Year * 100) + Week AS FiscalWeekKey
      ,OrgID
      ,UPPER(EntGradeGroup) EntGradeGroup
      ,ReceivedAmount AS ReceivedWgt  --Nucor uses Amount for Weights
      ,ReceivedValue AS ReceivedAmount -- Nucor uses Amount for dollar Amount
      
  FROM {federated_ods_catalog}.{SourceSchemaName}.ODS_NUE_RawMaterialReceipt
  WHERE DJJDeletedFlag=0 AND
  NOT (ReceivedAmount=0 AND
		ReceivedValue=0)

--and 1=2
""")

In [0]:
# ===================================================
# OLE_SRC_rawMatInventory - OLE DB Source
# ===================================================

ole_src_rawMatInventory_output_df = spark.sql(f"""
    SELECT (Year * 100) + Week AS FiscalWeekKey,
        OrgID,
        UPPER(EntGradeGroup) EntGradeGroup,
        InvBeginAmount AS BeginningWgt,
        InvBeginValue AS BeginningAmount,
        InvEndAmount AS EndingWgt,
        InvEndValue AS EndingAmount,
        InvConsumedAmount AS ConsumedWgt,
        InvConsumedValue AS ConsumedAmount,
	    InvBeginInTransitGTs AS BeginningInTransitWgt,
	    InvBeginInTransitDollars AS BeginningInTransitDollars,
	    InvEndInTransitGTs AS EndingInTransitWgt,
	    InvEndInTransitDollars AS EndingInTransitDollars
  FROM {federated_ods_catalog}.{SourceSchemaName}.ODS_NUE_RawMatInventory
  WHERE DJJDeletedFlag=0 AND 
		NOT (InvBeginValue=0 AND
			InvBeginAmount=0 AND
			InvEndValue=0 AND
			InvEndAmount=0 AND
			InvConsumedValue=0 AND
			InvConsumedAmount=0 AND
			InvBeginInTransitGTs=0 AND 
			InvBeginInTransitDollars=0 AND
			InvEndInTransitGTs=0 AND 
			InvEndInTransitDollars=0
			) 

--and year=2021 and week=20
""")

In [0]:
# =====================
# DER Inventory
# =====================

der_inventory_output_df = ole_src_rawMaterialReceipt_output_df \
                            .withColumn("d_BeginningWgt", F.lit(0).cast(IntegerType())) \
                            .withColumn("d_BeginningAmount", F.lit(0.0000).cast(DecimalType(18,4))) \
                            .withColumn("d_ConsumedWgt", F.lit(0).cast(IntegerType())) \
                            .withColumn("d_ConsumedAmount", F.lit(0.0000).cast(DecimalType(18,4))) \
                            .withColumn("d_EndingWgt", F.lit(0).cast(IntegerType())) \
                            .withColumn("d_EndingAmount", F.lit(0.0000).cast(DecimalType(18,4))) \
                            .withColumn("d_BeginningInTransitWgt", F.lit(0).cast(IntegerType())) \
                            .withColumn("d_BeginningInTransitDollars", F.lit(0.0000).cast(DecimalType(18,4))) \
                            .withColumn("d_EndingInTransitWgt", F.lit(0).cast(IntegerType())) \
                            .withColumn("d_EndingInTransitDollars", F.lit(0.0000).cast(DecimalType(18,4)))

In [0]:
# =====================
# DER Received
# =====================

der_received_output_df = ole_src_rawMatInventory_output_df \
                            .withColumn("d_ReceivedWgt", F.lit(0).cast(IntegerType())) \
                            .withColumn("d_ReceivedAmount", F.lit(0.00).cast(DecimalType(18,2)))

In [0]:
# ==================================
# DER Convert ReceiptWgt to LBs
# ==================================

der_convert_receiptWgt_output_df = der_inventory_output_df \
                            .withColumn("d_ReceivedWgt", (F.col('ReceivedWgt') * 2240).cast(IntegerType()))

In [0]:
# =============================
# DER Convert InvWgts to LBs
# =============================

der_convert_invwgts_output_df = der_received_output_df \
                            .withColumn("d_BeginningWgt", (F.col('BeginningWgt') * 2240).cast(IntegerType())) \
                            .withColumn("d_ConsumedWgt", (F.col('ConsumedWgt') * 2240).cast(IntegerType())) \
                            .withColumn("d_EndingWgt", (F.col('EndingWgt') * 2240).cast(IntegerType())) \
                            .withColumn("d_BeginningInTransitWgt", (F.col('BeginningInTransitWgt') * 2240).cast(IntegerType())) \
                            .withColumn("d_EndingInTransitWgt", (F.col('EndingInTransitWgt') * 2240).cast(IntegerType()))

In [0]:
# ==============================
# UNION Receipts & Inventory
# ==============================

union_output_df = der_convert_receiptWgt_output_df.select(
    F.col('OrgID'),
    F.col('EntGradeGroup'),
    F.col('FiscalWeekKey'),
    F.col('BeginningAmount'),
    F.col('EndingAmount'),
    F.col('ConsumedAmount'),
    F.col('d_ReceivedWgt'),
    F.col('d_EndingWgt'),
    F.col('d_ConsumedWgt'),
    F.col('d_BeginningWgt'),
    F.col('d_ReceivedAmount'),
    F.col('d_BeginningInTransitWgt'),
    F.col('d_EndingInTransitWgt'),
    F.col('EndingInTransitDollars').alias('d_EndingInTransitDollars'),
    F.col('BeginningInTransitDollars').alias('d_BeginningInTransitDollars')
).unionAll(der_convert_invwgts_output_df.select(
    F.col('OrgID'),
    F.col('EntGradeGroup'),
    F.col('FiscalWeekKey'),
    F.col('d_BeginningAmount').alias('BeginningAmount'),
    F.col('d_EndingAmount').alias('EndingAmount'),
    F.col('d_ConsumedAmount').alias('ConsumedAmount'),
    F.col('d_ReceivedWgt'),
    F.col('d_EndingWgt'),
    F.col('d_ConsumedWgt'),
    F.col('d_BeginningWgt'),
    F.col('ReceivedAmount').alias('d_ReceivedAmount'),
    F.col('d_BeginningInTransitWgt'),
    F.col('d_EndingInTransitWgt'),
    F.col('d_EndingInTransitDollars'),
    F.col('d_BeginningInTransitDollars')
)
)

In [0]:
# =======================
# DCNV EntGradeGroup
# =======================

dcnv_entGradeGroup = union_output_df \
                        .withColumn("c_EntGradeGroup", F.col('EntGradeGroup').cast(StringType()))

In [0]:
# =====================================================
# csplit_2BUNDLES - Conditional Split Transformation
# =====================================================

csplit_2BUNDLES_output_df = dcnv_entGradeGroup.filter(F.col("c_EntGradeGroup") == "#2 BUNDLES")
csplit_2BUNDLES_default_output_df = dcnv_entGradeGroup.filter(F.col("c_EntGradeGroup") != "#2 BUNDLES")

In [0]:
# ==============================================================
# DER_EntGradeGroup_2Bundles - Derived Column Transformation
# ==============================================================

der_EntGradeGroup_2Bundles_output_df = csplit_2BUNDLES_output_df \
    .withColumn(
        "c_EntGradeGroup",
        F.when(
            (F.col("c_EntGradeGroup") == "#2 BUNDLES") & (F.col("OrgID") == 37),
            "#2 BUNDLES-SECONDARIES"
        ).otherwise("#2 BUNDLES-PRIMES").cast(StringType())
    )

In [0]:
# ====================
# UNION 2BUNDLES
# ====================

union_2bundles_output_df = csplit_2BUNDLES_default_output_df.select(
    'OrgID',
    'EntGradeGroup',
    'FiscalWeekKey',
    'BeginningAmount',
    'EndingAmount',
    'ConsumedAmount',
    'd_ReceivedWgt',
    'd_EndingWgt',
    'd_ConsumedWgt',
    'd_BeginningWgt',
    'd_ReceivedAmount',
    'c_EntGradeGroup',
    'd_BeginningInTransitWgt',
    'd_EndingInTransitWgt',
    'd_EndingInTransitDollars',
    'd_BeginningInTransitDollars'
).unionAll(der_EntGradeGroup_2Bundles_output_df.select(
    'OrgID',
    'EntGradeGroup',
    'FiscalWeekKey',
    'BeginningAmount',
    'EndingAmount',
    'ConsumedAmount',
    'd_ReceivedWgt',
    'd_EndingWgt',
    'd_ConsumedWgt',
    'd_BeginningWgt',
    'd_ReceivedAmount',
    'c_EntGradeGroup',
    'd_BeginningInTransitWgt',
    'd_EndingInTransitWgt',
    'd_EndingInTransitDollars',
    'd_BeginningInTransitDollars'
    )
)

In [0]:
# ======================================================================
# LKP_EnterpriseGradeGroupKey - Standard Lookup Transformation
# ======================================================================

lkp_enterprise_grade_group_key_ref_df = spark.sql(f"""
                            SELECT EnterpriseGradeGroupKey, 
                                UPPER(EnterpriseGradeGroup) EnterpriseGradeGroup 
                            FROM {federated_starschema_catalog}.Brk.dimEnterpriseGradeGroup
                        """)

lkp_enterprise_grade_group_key_joined_df = union_2bundles_output_df.alias("inp").join(
    lkp_enterprise_grade_group_key_ref_df.alias("lkp"),
    F.col("inp.c_EntGradeGroup") == F.col("lkp.EnterpriseGradeGroup"),
    "left"
).select(
    F.col("inp.*"),
    F.col("lkp.EnterpriseGradeGroupKey").alias("EnterpriseGradeGroupKey")
)

In [0]:
# ======================================================================
# LKP_ConsumerNum - Standard Lookup Transformation
# ======================================================================

lkp_consumer_num_ref_df = spark.sql(f"""
            SELECT CAST(Code AS INT) AS OrgID, CAST(DJJConsumerNumber AS STRING) AS ConsumerNum 
            FROM {federated_ods_catalog}.{SourceSchemaName}.ODS_MDS_NUEMillNames 
            WHERE DJJDeletedFlag=0
        """)

lkp_consumer_num_joined_df = lkp_enterprise_grade_group_key_joined_df.alias("inp").join(
    lkp_consumer_num_ref_df.alias("lkp"),
    F.col("inp.OrgID") == F.col("lkp.OrgID"),
    "left"
).select(
    F.col("inp.*"),
    F.col("lkp.ConsumerNum").alias("ConsumerNum")
)

In [0]:
# ======================================================================
# LKP_ConsumerKey - Standard Lookup Transformation
# ======================================================================

lkp_consumer_key_ref_df = spark.sql(f"""
                            SELECT ConsumerKey, ConsumerID 
                            FROM {federated_starschema_catalog}.Brk.dimConsumer
                        """)

lkp_consumer_key_joined_df = lkp_enterprise_grade_group_key_joined_df.alias("inp").join(
    lkp_consumer_key_ref_df.alias("lkp"),
    F.col("inp.ConsumerNum") == F.col("lkp.ConsumerID"),
    "left"
).select(
    F.col("inp.*"),
    F.col("lkp.ConsumerKey").alias("ConsumerKey")
)

In [0]:
# ===============
# Aggregate
# ===============

aggregated_output_df = lkp_consumer_key_joined_df.groupBy(
    'FiscalWeekKey', 'OrgID', 'ConsumerKey', 'EnterpriseGradeGroupKey', 'c_EntGradeGroup', 'ConsumerNum'
).agg(
    F.sum('BeginningAmount').alias('BeginningAmount'),
    F.sum('EndingAmount').alias('EndingAmount'),
    F.sum('ConsumedAmount').alias('ConsumedAmount'),
    F.sum('d_ReceivedWgt').alias('d_ReceivedWgt'),
    F.sum('d_EndingWgt').alias('d_EndingWgt'),
    F.sum('d_ConsumedWgt').alias('d_ConsumedWgt'),
    F.sum('d_BeginningWgt').alias('d_BeginningWgt'),
    F.sum('d_ReceivedAmount').alias('d_ReceivedAmount'),
    F.sum('d_BeginningInTransitWgt').alias('d_BeginningInTransitWgt'),
    F.sum('d_EndingInTransitWgt').alias('d_EndingInTransitWgt'),
    F.sum('d_EndingInTransitDollars').alias('d_EndingInTransitDollars'),
    F.sum('d_BeginningInTransitDollars').alias('d_BeginningInTransitDollars')
)

In [0]:
# ================================================================
# LKP MilInTransit Adjustments - Standard Lookup Transformation
# ================================================================

lkp_milintransit_adjustments_ref_df = spark.sql(f"""
                SELECT (FiscalYear * 100) + FiscalWeek AS FiscalWeekKey, ConsumerID, 
                    CAST(UPPER(EntGradeGroup) AS VARCHAR(100)) AS EntGradeGroup, BeginningInTransitWgtGTs * 2240.0 AS BeginningInTransitWgtGTs,BeginningInTransitAmount,
                    EndingInTransitWgtGTs * 2240.0 AS EndingInTransitWgtGTs,
                    EndingInTransitAmount,
                    AdjustmentType
                FROM {federated_ods_catalog}.{SourceSchemaName}.ODS_NUE_MillInTransitAdjustment            
            """)

lkp_milintransit_adjustments_joined_df = aggregated_output_df.alias('inp').join(
    lkp_milintransit_adjustments_ref_df.alias('lkp'),
    (F.col("inp.FiscalWeekKey") == F.col("lkp.FiscalWeekKey")) & (F.col("inp.c_EntGradeGroup") == F.col("lkp.EntGradeGroup")) & (F.col("inp.ConsumerNum") == F.col("lkp.ConsumerID")),
    "left"
).select(
    F.col("inp.*"),
    F.col("lkp.BeginningInTransitWgtGTs").alias("BeginningInTransitWgtGTs_ADJ"),
    F.col("lkp.BeginningInTransitAmount").alias("BeginningInTransitAmount_ADJ"),
    F.col("lkp.EndingInTransitWgtGTs").alias("EndingInTransitWgtGTs_ADJ"),
    F.col("lkp.EndingInTransitAmount").alias("EndingInTransitAmount_ADJ"),
    F.col("lkp.AdjustmentType")
)

In [0]:
# =========================================================
# DER Adjusted InTransit - Derived Column Transformation
# =========================================================

der_adjusted_intransit_output_df = (
    lkp_milintransit_adjustments_joined_df
    .withColumn(
        "Adjusted_BeginInTransitWgt",
        F.when(
            F.col("BeginningInTransitWgtGTs_ADJ").isNull(),
            F.col("d_BeginningInTransitWgt")
        )
        .when(
            F.col("AdjustmentType") == "Replacement",
            F.col("BeginningInTransitWgtGTs_ADJ")
        )
        .otherwise(
            F.col("d_BeginningInTransitWgt") + F.col("BeginningInTransitWgtGTs_ADJ")
        ).cast(DecimalType(25,5))
    )
    .withColumn(
        "Adjusted_BeginInTransitAmount",
        F.when(
            F.col("BeginningInTransitDollars_ADJ").isNull(),
            F.col("d_BeginningInTransitDollars")
        )
        .when(
            F.col("AdjustmentType") == "Replacement",
            F.col("BeginningInTransitDollars_ADJ")
        )
        .otherwise(
            F.col("d_BeginningInTransitDollars") + F.col("BeginningInTransitDollars_ADJ")
        ).cast(DecimalType(19,4))
    )
    .withColumn(
        "Adjusted_EndingInTransitWgt",
        F.when(
            F.col("EndingInTransitWgtGTs_ADJ").isNull(),
            F.col("d_EndingInTransitWgt")
        )
        .when(
            F.col("AdjustmentType") == "Replacement",
            F.col("EndingInTransitWgtGTs_ADJ")
        )
        .otherwise(
            F.col("d_EndingInTransitWgt") + F.col("EndingInTransitWgtGTs_ADJ")
        ).cast(DecimalType(25,5))
    )
    .withColumn(
        "Adjusted_EndingInTransitAmount",
        F.when(
            F.col("EndingInTransitDollars_ADJ").isNull(),
            F.col("d_EndingInTransitDollars")
        )
        .when(
            F.col("AdjustmentType") == "Replacement",
            F.col("EndingInTransitDollars_ADJ")
        )
        .otherwise(
            F.col("d_EndingInTransitDollars") + F.col("EndingInTransitDollars_ADJ")
        ).cast(DecimalType(19,4))
    )
)

In [0]:
# =========================================================
# DER nulls - Derived Column Transformation
# =========================================================

der_nulls_output_df = der_adjusted_intransit_output_df \
    .withColumn(
        "EnterpriseGradeGroupKey", 
        F.coalesce(EnterpriseGradeGroupKey, -1).cast(IntegerType())
    ) \
    .withColumn(
        "ConsumerKey", 
        F.coalesce(ConsumerKey, -1).cast(IntegerType())
    )

In [0]:
# =========================================================
# DER Metadata - Derived Column Transformation
# =========================================================

der_metadata_output_df = der_nulls_output_df \
    .withColumn("DJJGeneratedFlag", F.lit(0).cast(IntegerType()) )
    .withColumn("ETLSSExecutionID", F.lit(None).cast(IntegerType()))
    .withColumn("DJJLastUpdateTime", F.lit('{etl_runtime}').cast(TimestampType()) )

# 4. TABLE LOAD (UPDATE / INSERT / DELETE)

In [0]:
# ============================================================
# UPDATE AND INSERT RECORDS
# ============================================================

final_df = der_metadata_output_df.select(
    F.col("FiscalWeekKey").cast(IntegerType()),
    F.col("OrgID").cast(IntegerType()),
    F.col("c_EntGradeGroup").cast(StringType()).alias("EnterpriseGradeGroup"),
    F.col("EnterpriseGradeGroupKey").cast(IntegerType()),
    F.col("ConsumerKey").cast(IntegerType()),
    F.col("d_ReceivedWgt").cast(LongType()).alias("ReceivedWgt"),
    F.col("d_ReceivedAmount").cast(DecimalType(18, 2)).alias("ReceivedAmount"),
    F.col("d_BeginningWgt").cast(LongType()).alias("BeginningWgt"),
    F.col("BeginningAmount").cast(DecimalType(18, 4)).alias("BeginningAmount"),
    F.col("d_ConsumedWgt").cast(LongType()).alias("ConsumedWgt"),
    F.col("ConsumedAmount").cast(DecimalType(18, 4)).alias("ConsumedAmount"),
    F.col("d_EndingWgt").cast(LongType()).alias("EndingWgt"),
    F.col("EndingAmount").cast(DecimalType(18, 4)).alias("EndingAmount"),
    F.col("Adjusted_BeginInTransitWgt").cast(IntegerType()).alias("BeginningInTransitWgt"),
    F.col("d_BeginningInTransitWgt").cast(IntegerType()).alias("BeginningInTransitWgt_BeforeAdjustment"),
    F.col("Adjusted_BeginInTransitAmount").cast(DecimalType(18, 4)).alias("BeginningInTransitAmount"),
    F.col("d_BeginningInTransitDollars").cast(DecimalType(18, 4)).alias("BeginningInTransitAmount_BeforeAdjustment"),
    F.col("Adjusted_EndingInTransitWgt").cast(IntegerType()).alias("EndingInTransitWgt"),
    F.col("d_EndingInTransitWgt").cast(IntegerType()).alias("EndingInTransitWgt_BeforeAdjustment"),
    F.col("Adjusted_EndingInTransitAmount").cast(DecimalType(18, 4)).alias("EndingInTransitAmount"),
    F.col("d_EndingInTransitDollars").cast(DecimalType(18, 4)).alias("EndingInTransitAmount_BeforeAdjustment"),
    F.col("DJJGeneratedFlag").cast(IntegerType()),
    F.col("ETLSSExecutionID").cast(LongType()),
    F.col("DJJLastUpdateTime").cast(TimestampType())
)

final_df.createOrReplaceTempView("merge_source_vw")

spark.sql(f"""
    MERGE INTO {target_table} AS target
    USING merge_source_vw AS source
    ON target.FiscalWeekKey = source.FiscalWeekKey
       AND target.OrgID = source.OrgID
       AND target.EnterpriseGradeGroup = source.EnterpriseGradeGroup

    WHEN MATCHED AND (
        target.EnterpriseGradeGroupKey <> source.EnterpriseGradeGroupKey OR
        target.ConsumerKey <> source.ConsumerKey OR
        target.ReceivedWgt <> source.ReceivedWgt OR
        target.ReceivedAmount <> source.ReceivedAmount OR
        target.BeginningWgt <> source.BeginningWgt OR
        target.BeginningAmount <> source.BeginningAmount OR
        target.ConsumedWgt <> source.ConsumedWgt OR
        target.ConsumedAmount <> source.ConsumedAmount OR
        target.EndingWgt <> source.EndingWgt OR
        target.EndingAmount <> source.EndingAmount OR
        target.BeginningInTransitWgt <> source.BeginningInTransitWgt OR
        target.BeginningInTransitWgt_BeforeAdjustment <> source.BeginningInTransitWgt_BeforeAdjustment OR
        target.BeginningInTransitAmount <> source.BeginningInTransitAmount OR
        target.BeginningInTransitAmount_BeforeAdjustment <> source.BeginningInTransitAmount_BeforeAdjustment OR
        target.EndingInTransitWgt <> source.EndingInTransitWgt OR
        target.EndingInTransitWgt_BeforeAdjustment <> source.EndingInTransitWgt_BeforeAdjustment OR
        target.EndingInTransitAmount <> source.EndingInTransitAmount OR
        target.EndingInTransitAmount_BeforeAdjustment <> source.EndingInTransitAmount_BeforeAdjustment
    )
    THEN UPDATE SET
        target.EnterpriseGradeGroupKey = source.EnterpriseGradeGroupKey,
        target.ConsumerKey = source.ConsumerKey,
        target.ReceivedWgt = source.ReceivedWgt,
        target.ReceivedAmount = source.ReceivedAmount,
        target.BeginningWgt = source.BeginningWgt,
        target.BeginningAmount = source.BeginningAmount,
        target.ConsumedWgt = source.ConsumedWgt,
        target.ConsumedAmount = source.ConsumedAmount,
        target.EndingWgt = source.EndingWgt,
        target.EndingAmount = source.EndingAmount,
        target.BeginningInTransitWgt = source.BeginningInTransitWgt,
        target.BeginningInTransitWgt_BeforeAdjustment = source.BeginningInTransitWgt_BeforeAdjustment,
        target.BeginningInTransitAmount = source.BeginningInTransitAmount,
        target.BeginningInTransitAmount_BeforeAdjustment = source.BeginningInTransitAmount_BeforeAdjustment,
        target.EndingInTransitWgt = source.EndingInTransitWgt,
        target.EndingInTransitWgt_BeforeAdjustment = source.EndingInTransitWgt_BeforeAdjustment,
        target.EndingInTransitAmount = source.EndingInTransitAmount,
        target.EndingInTransitAmount_BeforeAdjustment = source.EndingInTransitAmount_BeforeAdjustment,
        target.DJJGeneratedFlag = source.DJJGeneratedFlag,
        target.ETLSSExecutionID = source.ETLSSExecutionID,
        target.EnrichedTimestampUTC = source.DJJLastUpdateTime

    WHEN NOT MATCHED
    THEN INSERT (
        FiscalWeekKey,
        OrgID,
        EnterpriseGradeGroup,
        EnterpriseGradeGroupKey,
        ConsumerKey,
        ReceivedWgt,
        ReceivedAmount,
        BeginningWgt,
        BeginningAmount,
        ConsumedWgt,
        ConsumedAmount,
        EndingWgt,
        EndingAmount,
        BeginningInTransitWgt,
        BeginningInTransitWgt_BeforeAdjustment,
        BeginningInTransitAmount,
        BeginningInTransitAmount_BeforeAdjustment,
        EndingInTransitWgt,
        EndingInTransitWgt_BeforeAdjustment,
        EndingInTransitAmount,
        EndingInTransitAmount_BeforeAdjustment,
        DJJGeneratedFlag,
        ETLSSExecutionID,
        EnrichedTimestampUTC
    )
    VALUES (
        source.FiscalWeekKey,
        source.OrgID,
        source.EnterpriseGradeGroup,
        source.EnterpriseGradeGroupKey,
        source.ConsumerKey,
        source.ReceivedWgt,
        source.ReceivedAmount,
        source.BeginningWgt,
        source.BeginningAmount,
        source.ConsumedWgt,
        source.ConsumedAmount,
        source.EndingWgt,
        source.EndingAmount,
        source.BeginningInTransitWgt,
        source.BeginningInTransitWgt_BeforeAdjustment,
        source.BeginningInTransitAmount,
        source.BeginningInTransitAmount_BeforeAdjustment,
        source.EndingInTransitWgt,
        source.EndingInTransitWgt_BeforeAdjustment,
        source.EndingInTransitAmount,
        source.EndingInTransitAmount_BeforeAdjustment,
        source.DJJGeneratedFlag,
        source.ETLSSExecutionID,
        source.DJJLastUpdateTime
    )
""")

# 5. FINISH METADATA LOGGING

In [0]:
merge_history = spark.sql(f"DESCRIBE HISTORY {enriched_catalog}.{DestinationSchemaName}.{DestinationTableName}").filter("operation = 'MERGE'").first()
metrics = merge_history["operationMetrics"]
RowsInserted = int(metrics.get("numTargetRowsInserted", 0))
RowsUpdated = int(metrics.get("numTargetRowsUpdated", 0))
RowsDeleted = int(metrics.get("numTargetRowsDeleted", 0))

print(f"MERGE completed - Inserted: {RowsInserted}, Updated: {RowsUpdated}, Deleted: {RowsDeleted}")

print("ETL factInventorySnapshot completed successfully.")

In [0]:
# ==================================
# FINISH METADATA LOGGING
# ==================================

metrics = {
    "rows_inserted": RowsInserted,
    "rows_updated": RowsUpdated,
    "rows_deleted": RowsDeleted
}

end_execution_run(run_ctx, status="SUCCESS", metrics=metrics)

In [0]:
# ==================================
# Cleanup: Drop temporary view
# ==================================

spark.catalog.dropTempView("merge_source_vw")

In [0]:
%sql
drop table if exists djj_enriched_non_prod.NUE.fact_InventorySnapshot_temp